# 🏠 House Price Prediction Engine - V3.0
### Advanced Feature Engineering & Production Architecture

This notebook implements the production-grade pipeline used in our API. It includes custom feature engineering, logarithmic target normalization, and optimized XGBoost hyperparameters.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from xgboost import XGBRegressor

print("✅ Environment Setup Complete")

## 🛠️ Custom Feature Engineering
We use a custom `FeatureEngineer` to ensure consistency between training and the real-time API.

In [ ]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        X = X.copy()
        # House Age
        yr_sold = X['YrSold'] if 'YrSold' in X.columns else 2010
        X['HouseAge'] = yr_sold - X['YearBuilt']
        # Total Living Area Interactions
        X['TotalSF'] = X['GrLivArea'] + X['TotalBsmtSF'].fillna(0)
        X['Qual_Area_Interact'] = X['OverallQual'] * X['GrLivArea']
        # Aggregate Bathrooms
        if 'FullBath' in X.columns and 'HalfBath' in X.columns:
            X['TotalBath'] = X['FullBath'] + (0.5 * X['HalfBath'])
        return X

print("✅ FeatureEngineer Defined")

## 📊 Data Loading & Preprocessing

In [ ]:
df = pd.read_csv('train.csv', engine='python')
df = df[df['GrLivArea'] < 4500]

BASE_FEATURES = [
    "OverallQual", "GrLivArea", "GarageCars", "TotalBsmtSF", 
    "FullBath", "YearBuilt", "Neighborhood", "MSZoning", "YrSold", "HalfBath"
]
X = df[BASE_FEATURES]
y = np.log1p(df['SalePrice'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

print(f"Data Split: {X_train.shape[0]} samples")

## 🏗️ The Production Pipeline

In [ ]:
num_cols = ["OverallQual", "GrLivArea", "GarageCars", "TotalBsmtSF", "FullBath", "YearBuilt", "HouseAge", "TotalSF", "Qual_Area_Interact"]
cat_cols = ["Neighborhood", "MSZoning"]

num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
], remainder='drop')

model = Pipeline(steps=[
    ('engineer', FeatureEngineer()),
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        n_estimators=1000, learning_rate=0.05, max_depth=5, 
        subsample=0.8, colsample_bytree=0.8, random_state=42
    ))
])

model.fit(X_train, y_train)
print("✅ Model V3.0 Trained Successfully")

## 📈 Performance Evaluation

In [ ]:
preds_log = model.predict(X_test)
y_true = np.expm1(y_test)
y_pred = np.expm1(preds_log)

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_test, preds_log)

print(f"MAE: ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")
print(f"Log-Scale R2: {r2:.4f}")

plt.figure(figsize=(8, 6))
sns.regplot(x=y_true, y=y_pred, scatter_kws={'alpha':0.5}, line_kws={'color':'red'})
plt.title("Actual vs Predicted House Prices (USD)")
plt.show()